# OnionEye · 04 · Train the defect classifier
T4 GPU, about 30 min. Output: `MyDrive/onioneye/models/classifier.onnx` + confusion matrix and scores in `reports/`.

In [ ]:
!pip -q install ultralytics onnx onnxruntime onnxslim scikit-learn

In [ ]:
# ---- setup: works on Google Colab (default) and Kaggle
import os, sys, glob, shutil, zipfile, random
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT, WORK = '/content/drive/MyDrive/onioneye', '/content'
elif IN_KAGGLE:
    ROOT, WORK = '/kaggle/working/onioneye', '/kaggle/temp'
else:
    ROOT, WORK = os.path.abspath('onioneye_data'), os.path.abspath('work')
for d in (ROOT, WORK, f'{ROOT}/models', f'{ROOT}/reports'):
    os.makedirs(d, exist_ok=True)
print('saving to', ROOT)

In [ ]:
CLS = f'{WORK}/defects'; HOLD = f'{WORK}/holdout'
if not os.path.exists(CLS): zipfile.ZipFile(f'{ROOT}/defect_dataset.zip').extractall(CLS)
if not os.path.exists(HOLD) and os.path.exists(f'{ROOT}/mendeley_unhealthy_holdout.zip'):
    zipfile.ZipFile(f'{ROOT}/mendeley_unhealthy_holdout.zip').extractall(HOLD)
from ultralytics import YOLO
RUNS = f'{ROOT}/runs'
last = f'{RUNS}/classifier/weights/last.pt'
if os.path.exists(last):
    YOLO(last).train(resume=True)
else:
    YOLO('yolo11n-cls.pt').train(data=CLS, imgsz=224, epochs=40, patience=10, batch=64,
                                 project=RUNS, name='classifier', exist_ok=True, seed=42,
                                 fliplr=0.5, flipud=0.5, degrees=20, hsv_h=0.01)

## 1 · Test-set scores (per class) and confusion matrix

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
best = YOLO(f'{RUNS}/classifier/weights/best.pt')
names = [best.names[i] for i in range(len(best.names))]
y_true, y_pred = [], []
for i, c in enumerate(names):
    files = glob.glob(f'{CLS}/test/{c}/*')
    for r in best.predict(files, imgsz=224, verbose=False, stream=True):
        y_true.append(i); y_pred.append(int(r.probs.top1))
rep = classification_report(y_true, y_pred, labels=list(range(len(names))), target_names=names, digits=3, output_dict=True, zero_division=0)
df = pd.DataFrame(rep).T.round(3); display(df); df.to_csv(f'{ROOT}/reports/classifier_scores.csv')
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=list(range(len(names)))), display_labels=names).plot(xticks_rotation=45)
plt.title('Defect classifier, test set'); plt.tight_layout(); plt.savefig(f'{ROOT}/reports/classifier_confusion.png', dpi=110); plt.show()

## 2 · Sanity check on 1,500 held-out Mendeley *unhealthy* bulbs (never trained on)

In [ ]:
files = glob.glob(f'{HOLD}/*.jpg')
if files:
    preds = [best.names[int(r.probs.top1)] for r in best.predict(files, imgsz=224, verbose=False, stream=True)]
    s = pd.Series(preds).value_counts(normalize=True).mul(100).round(1)
    print(f"called NOT good: {100 - s.get('good', 0):.1f}% (higher is better)"); display(s)

## 3 · Export for the backend

In [ ]:
shutil.copy(f'{RUNS}/classifier/weights/best.pt', f'{ROOT}/models/classifier_best.pt')
onnx_path = best.export(format='onnx', imgsz=224, opset=12, simplify=True)
shutil.copy(onnx_path, f'{ROOT}/models/classifier.onnx')
import onnxruntime as ort
s = ort.InferenceSession(onnx_path)
print(s.get_outputs()[0].shape, s.get_modelmeta().custom_metadata_map.get('names'))

Done ✅ Download `models/classifier.onnx` → `backend/models/`, restart the backend.